# Phasing Benchmarks on NA12878 Chromosome 22 (PacBio HiFi)

Ce notebook teste et compare l'algorithme de dynamique géométrique de Glauber (MCMC) contre la baseline spectrale ainsi que l'outil de l'état de l'art **WhatsHap** sur le chromosome 22 du génome réel de référence **NA12878**.

## Objectifs
1. Télécharger et extraire à distance les données réelles (BAM PacBio HiFi, BED et VCF de vérité terrain) pour le chromosome 22.
2. Construire un graphe d'interactions de reads signé et pondéré selon les formules bayésiennes avec prise en compte des scores de qualité Phred de séquençage.
3. Appliquer un facteur de sécurité $\beta = 1/2$ sur les poids des arêtes.
4. Lancer la dynamique de Glauber (MCMC) et le clustering spectral signed Laplacian pour prédire les phases.
5. Reconstruire la phase des variants et exporter au format VCF.
6. Lancer **WhatsHap** et faire une comparaison quantitative complète via `whatshap compare` (Switch Error Rate, Hamming distance, Block N50).

In [ ]:
# @title Installation des outils système et des dépendances Python
import sys
import os

print("⏳ Installation de samtools et tabix...")
!apt-get update && apt-get install -y samtools tabix

print("⏳ Installation des bibliothèques Python...")
!pip install -q numba scipy pandas matplotlib pysam whatshap

print("✅ Installations terminées !")

In [ ]:
# @title Téléchargement et extraction des données de NA12878 (PacBio HiFi)
import urllib.request
import pysam
import os
chromosome = "chr22"

VCF_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/NA12878_HG001/NISTv4.2.1/GRCh38/SupplementaryFiles/HG001_GRCh38_1_22_v4.2.1_benchmark_hifiasm_v11_phasetransfer.vcf.gz"
TBI_URL = VCF_URL + ".tbi"
BED_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/NA12878_HG001/NISTv4.2.1/GRCh38/HG001_GRCh38_1_22_v4.2.1_benchmark.bed"
BAM_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/data/NA12878/PacBio_SequelII_CCS_11kb/HG001_GRCh38/HG001_GRCh38.haplotag.RTG.trio.bam"

print("🧬 Téléchargement du VCF phased et de son index (130 Mo)...")
urllib.request.urlretrieve(VCF_URL, "HG001_phased.vcf.gz")
urllib.request.urlretrieve(TBI_URL, "HG001_phased.vcf.gz.tbi")

print("🧬 Téléchargement du BED de confiance...")
urllib.request.urlretrieve(BED_URL, "HG001_benchmark.bed")

print("📥 Extraction du BAM d'alignement PacBio HiFi pour chr22 (cette étape prend ~1 minute)...")
pysam.samtools.view("-h", "-b", BAM_URL, "chr22", "-o", "NA12878_chr22_alignment.bam", catch_stdout=False)
pysam.samtools.index("NA12878_chr22_alignment.bam")

print("🧬 Création des fichiers VCF nettoyés pour chr22 (déphasé pour prédictions, et phased pour vérité terrain)...")
def prepare_cleaned_vcfs(input_phased_path, output_unphased_path, output_phased_chr22_path, chromosome="chr22"):
    with pysam.VariantFile(input_phased_path) as in_vcf:
        header = pysam.VariantHeader()
        for s in in_vcf.header.samples:
            header.add_sample(s)
        for r in in_vcf.header.records:
            if r.key == 'FORMAT' and r.get('ID') == 'PS':
                header.add_line('##FORMAT=<ID=PS,Number=1,Type=Integer,Description="Phase set">')
            elif r.key in ['fileformat', 'FILTER', 'FORMAT', 'INFO', 'contig']:
                header.add_record(r)
        with pysam.VariantFile(output_unphased_path, "w", header=header) as out_unphased, \
             pysam.VariantFile(output_phased_chr22_path, "w", header=header) as out_phased:
            for record in in_vcf.fetch(chromosome):
                rec_phased = header.new_record(
                    contig=record.chrom, start=record.start, stop=record.stop,
                    alleles=record.alleles, id=record.id, qual=record.qual, filter=record.filter.keys()
                )
                for sample_name in record.samples:
                    in_sample = record.samples[sample_name]
                    out_sample = rec_phased.samples[sample_name]
                    for fmt_key in header.formats:
                        if fmt_key == 'PS':
                            ps_val = in_sample.get('PS')
                            if ps_val is not None:
                                try:
                                    out_sample['PS'] = int(ps_val)
                                except ValueError:
                                    out_sample['PS'] = 1
                        else:
                            if fmt_key in in_sample:
                                out_sample[fmt_key] = in_sample[fmt_key]
                    out_sample.phased = in_sample.phased
                out_phased.write(rec_phased)
                rec_unphased = header.new_record(
                    contig=record.chrom, start=record.start, stop=record.stop,
                    alleles=record.alleles, id=record.id, qual=record.qual, filter=record.filter.keys()
                )
                for sample_name in record.samples:
                    in_sample = record.samples[sample_name]
                    out_sample = rec_unphased.samples[sample_name]
                    for fmt_key in header.formats:
                        if fmt_key == 'PS':
                            out_sample['PS'] = None
                        else:
                            if fmt_key in in_sample:
                                out_sample[fmt_key] = in_sample[fmt_key]
                    if out_sample.allele_indices is not None:
                        out_sample.phased = False
                out_unphased.write(rec_unphased)
prepare_cleaned_vcfs("HG001_phased.vcf.gz", "HG001_unphased.vcf", "HG001_phased_chr22.vcf", chromosome)

print("✅ Téléchargement et extraction terminés avec succès !")


In [ ]:
# @title 🎛️ Formulaire de Configuration des Paramètres Globaux { display-mode: "form" }

chromosome = "chr22" #@param ["chr1", "chr2", "chr20", "chr22"] {type:"string"}
beta_safety = 0.5 #@param {type:"number"}
k_hop = 3 #@param {type:"integer"}
mcmc_steps = 20000000 #@param {type:"integer"}
beta = 1.0 #@param {type:"number"}
verbose = True #@param {type:"boolean"}

print("✨ Paramètres configurés avec succès !")
print(f"  - Chromosome cible : {chromosome}")
print(f"  - k-hop de MCMC : {k_hop}")
print(f"  - Nombre de pas MCMC : {mcmc_steps}")
print(f"  - Température inverse (beta) : {beta}")
print(f"  - Mode verbeux (verbose) : {verbose}")


In [ ]:
# @title Importations et vérification du GPU
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import numba
import time
import matplotlib.pyplot as plt
import pysam
import os
from collections import defaultdict

try:
    import cupy as cp
    import cupyx.scipy.sparse as csp
    from cupyx.scipy.sparse.linalg import eigsh as cupy_eigsh
    GPU_AVAILABLE = cp.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"GPU disponible via CuPy! Solveur GPU activé.")
    else:
        print("CuPy importé, mais aucun GPU n'a été détecté. Repli sur le CPU.")
except ImportError:
    GPU_AVAILABLE = False
    print("CuPy non installé. Repli sur le CPU.")

In [ ]:
# @title Extraction des variants hétérozygotes et des profils des reads via WhatsHap
def load_variants(vcf_path, bed_path, chromosome="chr22"):
    # Charger les régions de confiance du BED
    import bisect
    bed_intervals = []
    with open(bed_path, "r") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 3 and parts[0] == chromosome:
                bed_intervals.append((int(parts[1]), int(parts[2])))
                
    # Trier les intervalles pour la recherche dichotomique
    bed_intervals.sort()
    
    def is_in_bed(pos):
        idx = bisect.bisect_right(bed_intervals, (pos, float("inf"))) - 1
        if idx >= 0:
            start, end = bed_intervals[idx]
            if start <= pos < end:
                return True
        return False
                
    variants = {}
    with pysam.VariantFile(vcf_path) as vcf:
        for record in vcf:
            if record.chrom != chromosome:
                continue
            pos = record.pos - 1
            if is_in_bed(pos):
                # Ne conserver que les variants bialléliques simples
                if len(record.ref) == 1 and len(record.alts) == 1 and len(record.alts[0]) == 1:
                    sample = list(record.samples.values())[0]
                    # S'assurer que le variant est hétérozygote dans la vérité terrain
                    if sample.phased and sample.allele_indices in [(0, 1), (1, 0)]:
                        variants[pos] = {
                            "ref": record.ref,
                            "alt": record.alts[0],
                            "gt": sample.allele_indices
                        }
    return variants

def extract_read_profiles_whatshap(bam_path, vcf_path, chromosome="chr22", verbose=False):
    from whatshap.cli import PhasedInputReader
    from whatshap.core import NumericSampleIds
    import time
    
    t_start = time.time()
    if verbose:
        print(f"⏳ [WhatsHap API] Initialisation du PhasedInputReader pour {bam_path} et {vcf_path}...")
        
    numeric_sample_ids = NumericSampleIds()
    inputs = [bam_path, vcf_path]
    
    with PhasedInputReader(
        bam_or_vcf_paths=inputs,
        reference=None,
        numeric_sample_ids=numeric_sample_ids,
        ignore_read_groups=True,
        only_snvs=True,
        mapq_threshold=20
    ) as reader:
        reader.read_vcfs()
        variant_table = reader._vcfs[0][chromosome]
        
        if verbose:
            print(f"  -> {len(variant_table.variants)} variants trouvés dans le VCF via le parser WhatsHap.")
            print(f"⏳ [WhatsHap API] Alignement local via PairHMM et calcul des vraisemblances pour {chromosome} (cette étape prend ~30 secondes)...")
            
        readset, vcf_source_ids = reader.read(chromosome, variant_table.variants, sample=None)
        
        if verbose:
            print(f"  -> {len(readset)} fragments de lecture (reads) uniques extraits du BAM par WhatsHap.")
            
        read_profiles = {}
        for read in readset:
            read_id = read.name
            profile = {}
            for var_call in read:
                pos = var_call.position
                allele = var_call.allele
                qual = var_call.quality if var_call.quality is not None else 30.0
                
                # Convertir le score Phred en probabilité d'erreur effective
                epsilon = max(10 ** (-qual / 10.0), 1e-4)
                profile[pos] = (allele, epsilon)
            if len(profile) > 0:
                read_profiles[read_id] = profile
                
    t_duration = time.time() - t_start
    if verbose:
        print(f"✅ [WhatsHap API] Extraction et réalignements terminés en {t_duration:.2f}s ({len(read_profiles)} reads actifs).")
        
    return read_profiles

print("⏳ Chargement des variants de vérité terrain...")
variants = load_variants("HG001_phased_chr22.vcf", "HG001_benchmark.bed", chromosome)
print(f"  -> {len(variants)} variants hétérozygotes bialléliques extraits.")

print("⏳ Extraction des profils des reads avec WhatsHap API...")
read_profiles = extract_read_profiles_whatshap("NA12878_chr22_alignment.bam", "HG001_unphased.vcf", chromosome, verbose=verbose)
if not verbose:
    print(f"  -> {len(read_profiles)} reads alignés extraits.")


In [ ]:
# @title Construction de la vérité terrain des reads et du graphe d'interactions signées
def get_true_read_spins(read_profiles, variants):
    true_spins = {}
    for read_id, profile in read_profiles.items():
        votes = []
        for pos, (allele_val, _) in profile.items():
            true_gt = variants[pos]["gt"]
            # true_gt[0] est l'allèle maternel (+1), true_gt[1] est l'allèle paternel (-1)
            if allele_val == true_gt[0]:
                votes.append(1)
            elif allele_val == true_gt[1]:
                votes.append(-1)
        if len(votes) > 0:
            true_spins[read_id] = 1 if np.sum(votes) >= 0 else -1
    return true_spins

print("⏳ Détermination des haplotypes d'origine des reads (vérité terrain)...")
true_spins = get_true_read_spins(read_profiles, variants)
active_reads = {rid: prof for rid, prof in read_profiles.items() if rid in true_spins}
# Tri explicite des reads par coordonnée génomique pour respecter la géométrie Z
read_ids = sorted(active_reads.keys(), key=lambda rid: min(active_reads[rid].keys()))
R = len(read_ids)
read_id_to_idx = {rid: i for i, rid in enumerate(read_ids)}
true_spin_vec = np.array([true_spins[rid] for rid in read_ids])

print(f"  -> {R} reads utilisables restants.")

print("⏳ Construction des arêtes et calcul des poids bayésiens (avec facteur de sécurité beta=0.5)...")
from collections import defaultdict

reads_per_variant = defaultdict(list)
for rid, profile in active_reads.items():
    for pos in profile.keys():
        reads_per_variant[pos].append(rid)

edges_dict = {}
for pos, rids in reads_per_variant.items():
    n_rids = len(rids)
    for i in range(n_rids):
        rid_i = rids[i]
        u = read_id_to_idx[rid_i]
        val_i, eps_i = active_reads[rid_i][pos]
        for j in range(i + 1, n_rids):
            rid_j = rids[j]
            v = read_id_to_idx[rid_j]
            val_j, eps_j = active_reads[rid_j][pos]
            
            edge_key = (u, v) if u < v else (v, u)
            
            # Probabilité d'accord sous le même haplotype
            q_z = (1 - eps_i) * (1 - eps_j) + eps_i * eps_j
            q_z = np.clip(q_z, 1e-6, 1.0 - 1e-6)
            log_ratio = np.log(q_z / (1.0 - q_z))
            
            is_concordant = (val_i == val_j)
            weight_contrib = log_ratio if is_concordant else -log_ratio
            
            if edge_key not in edges_dict:
                edges_dict[edge_key] = [0, 0, 0.0]
            
            if is_concordant:
                edges_dict[edge_key][0] += 1
            else:
                edges_dict[edge_key][1] += 1
            edges_dict[edge_key][2] += weight_contrib

edges_list = []
# beta_safety est défini dans le formulaire de configuration globale

for (u, v), (concordances, differences, weight_sum) in edges_dict.items():
    shared = concordances + differences
    if shared >= 1:
        scaled_weight = weight_sum * beta_safety
        sign = 1 if scaled_weight >= 0 else -1
        edges_list.append({
            "source": u,
            "target": v,
            "shared_het_sites": shared,
            "concordances": concordances,
            "differences": differences,
            "weight": scaled_weight,
            "sign": sign
        })

edges_df = pd.DataFrame(edges_list)
if verbose:
    if len(edges_list) > 0:
        print(f"  -> Nombre de votes concordants : {sum(e['concordances'] for e in edges_list)}")
        print(f"  -> Nombre de votes discordants : {sum(e['differences'] for e in edges_list)}")
        weights = [e['weight'] for e in edges_list]
        print(f"  -> Poids absolu moyen des arêtes : {np.mean(np.abs(weights)):.4f}")
        print(f"  -> Poids max positif : {max(weights):.4f}, Poids min négatif : {min(weights):.4f}")
    else:
        print("  -> Graphe vide (aucune arête).")
print(f"  -> Graphe construit : {R} sommets (reads) et {len(edges_df)} arêtes.")

In [ ]:
# @title Algorithmes d'accélération (Numba) et Génération de paires k-hop
@numba.njit
def build_cross_arrays_numba(R, left, right, weight):
    cut_counts = np.zeros(R, dtype=np.int32)
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        for q in range(u, v):
            cut_counts[q] += 1
            
    cross_offsets = np.zeros(R, dtype=np.int32)
    for q in range(R - 1):
        cross_offsets[q+1] = cross_offsets[q] + cut_counts[q]
        
    total_crossings = cross_offsets[R-1]
    cross_left = np.empty(total_crossings, dtype=np.int32)
    cross_right = np.empty(total_crossings, dtype=np.int32)
    cross_weight = np.empty(total_crossings, dtype=np.float64)
    
    current_idx = cross_offsets.copy()
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        w = weight[idx]
        for q in range(u, v):
            write_pos = current_idx[q]
            cross_left[write_pos] = u
            cross_right[write_pos] = v
            cross_weight[write_pos] = w
            current_idx[q] += 1
            
    return cross_offsets, cross_left, cross_right, cross_weight

@numba.njit
def build_incident_arrays_numba(R, left, right, weight):
    node_counts = np.zeros(R + 1, dtype=np.int32)
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        node_counts[u] += 1
        node_counts[v] += 1
        
    incident_offsets = np.zeros(R + 1, dtype=np.int32)
    for r in range(R):
        incident_offsets[r+1] = incident_offsets[r] + node_counts[r]
        
    total_incident = incident_offsets[R]
    incident_left = np.empty(total_incident, dtype=np.int32)
    incident_right = np.empty(total_incident, dtype=np.int32)
    incident_weight = np.empty(total_incident, dtype=np.float64)
    
    current_idx = incident_offsets.copy()
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        w = weight[idx]
        
        pos_u = current_idx[u]
        incident_left[pos_u] = u
        incident_right[pos_u] = v
        incident_weight[pos_u] = w
        current_idx[u] += 1
        
        pos_v = current_idx[v]
        incident_left[pos_v] = u
        incident_right[pos_v] = v
        incident_weight[pos_v] = w
        current_idx[v] += 1
        
    return incident_offsets, incident_left, incident_right, incident_weight

def build_structures_fast(R, edges_df):
    left = np.minimum(edges_df['source'], edges_df['target']).values.astype(np.int32)
    right = np.maximum(edges_df['source'], edges_df['target']).values.astype(np.int32)
    weight = edges_df['weight'].values.astype(np.float64)
    
    cross_offsets, cross_left, cross_right, cross_weight = build_cross_arrays_numba(R, left, right, weight)
    incident_offsets, incident_left, incident_right, incident_weight = build_incident_arrays_numba(R, left, right, weight)
    
    return (cross_offsets, cross_left, cross_right, cross_weight,
            incident_offsets, incident_left, incident_right, incident_weight)

def build_pairs_sparse(R, edges_df, k, gpu=True):
    row = np.concatenate([edges_df['source'].values, edges_df['target'].values])
    col = np.concatenate([edges_df['target'].values, edges_df['source'].values])
    data = np.ones(len(row), dtype=np.bool_)
    
    if gpu and GPU_AVAILABLE:
        try:
            row_gpu = cp.array(row)
            col_gpu = cp.array(col)
            data_gpu = cp.array(data)
            A_gpu = csp.coo_matrix((data_gpu, (row_gpu, col_gpu)), shape=(R, R)).tocsr()
            I_gpu = csp.eye(R, dtype=cp.bool_, format='csr')
            Visited_gpu = (A_gpu + I_gpu)
            current_gpu = Visited_gpu
            for _ in range(k - 1):
                current_gpu = current_gpu @ Visited_gpu
            current_tri_gpu = csp.triu(current_gpu, k=1)
            coo_gpu = current_tri_gpu.tocoo()
            return cp.asnumpy(coo_gpu.row).astype(np.int32), cp.asnumpy(coo_gpu.col).astype(np.int32)
        except Exception as e:
            print(f"[build_pairs_sparse GPU] Échec : {e}. Repli sur le CPU...")
    
    A = sp.coo_matrix((data, (row, col)), shape=(R, R)).tocsr()
    I = sp.eye(R, dtype=np.bool_, format='csr')
    
    Visited = (A + I)
    current_m = Visited
    for _ in range(k - 1):
        current_m = current_m @ Visited
        
    current_tri = sp.triu(current_m, k=1)
    coo = current_tri.tocoo()
    return coo.row.astype(np.int32), coo.col.astype(np.int32)

In [ ]:
# @title Fenwick Tree, MCMC Loop et Solveur Laplacien
@numba.njit
def fenwick_update(tree, idx, val):
    i = idx + 1
    n = len(tree)
    while i < n:
        tree[i] ^= val
        i += i & (-i)

@numba.njit
def fenwick_query(tree, idx):
    if idx < 0:
        return 0
    i = idx + 1
    res = 0
    while i > 0:
        res ^= tree[i]
        i -= i & (-i)
    return res

@numba.njit
def evaluate_cut(q, tree, cross_offsets, cross_left, cross_right, cross_weight):
    start = cross_offsets[q]
    end = cross_offsets[q+1]
    du = 0.0
    for idx in range(start, end):
        i = cross_left[idx]
        j = cross_right[idx]
        w = cross_weight[idx]
        
        xor_val = fenwick_query(tree, j-1) ^ fenwick_query(tree, i-1)
        spin_prod = 1.0 - 2.0 * float(xor_val)
        
        du += w * spin_prod
    return du

@numba.njit
def evaluate_singleton(r, tree, incident_offsets, incident_left, incident_right, incident_weight):
    start = incident_offsets[r]
    end = incident_offsets[r+1]
    du = 0.0
    for idx in range(start, end):
        i = incident_left[idx]
        j = incident_right[idx]
        w = incident_weight[idx]
        
        xor_val = fenwick_query(tree, j-1) ^ fenwick_query(tree, i-1)
        spin_prod = 1.0 - 2.0 * float(xor_val)
        
        du += w * spin_prod
    return du

@numba.njit
def mcmc_loop(steps, R, beta, tree, 
              cross_offsets, cross_left, cross_right, cross_weight,
              incident_offsets, incident_left, incident_right, incident_weight,
              pairs_left, pairs_right, verbose=False):
    
    # Pré-allocation des tableaux d'événements
    max_flips = 2 * steps
    flip_steps = np.empty(max_flips, dtype=np.int32)
    flip_walls = np.empty(max_flips, dtype=np.int32)
    flip_counts = np.zeros(R - 1, dtype=np.int32)
    total_flips = 0
    report_interval = steps // 10
    if report_interval == 0: report_interval = 1
    
    for t in range(1, steps + 1):
        r = np.random.randint(0, R)
        
        du0 = 0.0
        du1 = evaluate_singleton(r, tree, incident_offsets, incident_left, incident_right, incident_weight)
        
        if r - 1 >= 0:
            du2 = evaluate_cut(r-1, tree, cross_offsets, cross_left, cross_right, cross_weight)
        else:
            du2 = 0.0
            
        if r < R - 1:
            du3 = evaluate_cut(r, tree, cross_offsets, cross_left, cross_right, cross_weight)
        else:
            du3 = 0.0
            
        min_du = min(du0, du1, du2, du3)
        w0 = np.exp(-beta * (du0 - min_du))
        w1 = np.exp(-beta * (du1 - min_du))
        w2 = np.exp(-beta * (du2 - min_du))
        w3 = np.exp(-beta * (du3 - min_du))
        
        sum_w = w0 + w1 + w2 + w3
        p0 = w0 / sum_w
        p1 = w1 / sum_w
        p2 = w2 / sum_w
        p3 = w3 / sum_w
        
        rand_val = np.random.random()
        chosen_move = 0
        if rand_val < p0:
            chosen_move = 0
        elif rand_val < p0 + p1:
            chosen_move = 1
        elif rand_val < p0 + p1 + p2:
            chosen_move = 2
        else:
            chosen_move = 3
            
        if verbose and t % report_interval == 0:
            print("  -> Progression MCMC : ", 100 * t // steps, "%")
        if chosen_move == 1:
            if r - 1 >= 0:
                fenwick_update(tree, r-1, 1)
                flip_steps[total_flips] = t
                flip_walls[total_flips] = r-1
                total_flips += 1
                flip_counts[r-1] += 1
            if r < R - 1:
                fenwick_update(tree, r, 1)
                flip_steps[total_flips] = t
                flip_walls[total_flips] = r
                total_flips += 1
                flip_counts[r] += 1
        elif chosen_move == 2:
            if r - 1 >= 0:
                fenwick_update(tree, r-1, 1)
                flip_steps[total_flips] = t
                flip_walls[total_flips] = r-1
                total_flips += 1
                flip_counts[r-1] += 1
        elif chosen_move == 3:
            if r < R - 1:
                fenwick_update(tree, r, 1)
                flip_steps[total_flips] = t
                flip_walls[total_flips] = r
                total_flips += 1
                flip_counts[r] += 1
                
    # Slicing des événements réels
    actual_flip_steps = flip_steps[:total_flips]
    actual_flip_walls = flip_walls[:total_flips]
    
    # Regroupement des événements par mur (Structure CSR-like)
    offsets = np.zeros(R, dtype=np.int32)
    for k in range(R - 1):
        offsets[k+1] = offsets[k] + flip_counts[k]
        
    grouped_steps = np.empty(total_flips, dtype=np.int32)
    current_idx = offsets.copy()
    for i in range(total_flips):
        k = actual_flip_walls[i]
        t = actual_flip_steps[i]
        pos = current_idx[k]
        grouped_steps[pos] = t
        current_idx[k] += 1
        
    # Calcul exact de la corrélation par intervalle de temps pour chaque paire k-hop
    P = len(pairs_left)
    correlations = np.empty(P, dtype=np.float64)
    
    for p in range(P):
        u = pairs_left[p]
        v = pairs_right[p]
        
        # Intervalle des parois de domaine [u, v-1]
        sum_counts = 0
        for k in range(u, v):
            sum_counts += flip_counts[k]
            
        if sum_counts == 0:
            correlations[p] = 1.0
            continue
            
        # Rassembler les événements pour l'intervalle
        temp = np.empty(sum_counts, dtype=np.int32)
        idx_temp = 0
        for k in range(u, v):
            start = offsets[k]
            count = flip_counts[k]
            for idx_grouped in range(start, start + count):
                temp[idx_temp] = grouped_steps[idx_grouped]
                idx_temp += 1
                
        # Trier les pas de temps
        temp = np.sort(temp)
        
        # Filtrer les occurrences impaires et intégrer sur le temps
        sum_prod = 0.0
        current_sign = 1.0  # Au départ, tous les spins sont à +1
        last_t = 1
        
        idx = 0
        while idx < sum_counts:
            t_val = temp[idx]
            cnt = 1
            while idx + 1 < sum_counts and temp[idx + 1] == t_val:
                cnt += 1
                idx += 1
            if cnt % 2 == 1:
                sum_prod += current_sign * (t_val - last_t)
                current_sign = -current_sign
                last_t = t_val
            idx += 1
            
        sum_prod += current_sign * (steps + 1 - last_t)
        correlations[p] = sum_prod / float(steps)
        
    return correlations

def solve_signed_spectral(W, gpu=True):
    R = W.shape[0]
    abs_W = abs(W)
    degrees = np.array(abs_W.sum(axis=1)).flatten()
    
    if gpu and GPU_AVAILABLE:
        try:
            W_gpu = csp.csr_matrix(W)
            D_gpu = csp.diags(cp.array(degrees))
            L_gpu = D_gpu - W_gpu
            vals, vecs = cupy_eigsh(L_gpu, k=1, which='SA')
            return cp.asnumpy(vecs[:, 0]), cp.asnumpy(vals[0])
        except Exception as e:
            if verbose: print(f"[Solveur GPU] Échec : {e}. Repli sur le CPU...")
            
    D = sp.diags(degrees)
    L = D - W
    try:
        sigma = 2.0 * np.max(degrees)
        I = sp.eye(R, format='csr')
        M = sigma * I - L
        vals, vecs = eigsh(M, k=1, which='LA')
        return vecs[:, 0], sigma - vals[0]
    except Exception as e:
        if verbose: print(f"[Solveur CPU] Échec : {e}. Repli sur le solveur SM standard...")
        vals, vecs = eigsh(L, k=1, which='SM')
        return vecs[:, 0], vals[0]


In [ ]:
# @title Exécution de la Baseline et du Glauber MCMC sur chr22
# 1. Baseline: Direct Signed Spectral Clustering sur le graphe initial
print("⏳ Exécution de la Baseline (Spectral direct)...")
t0 = time.time()
row = np.concatenate([edges_df['source'].values, edges_df['target'].values])
col = np.concatenate([edges_df['target'].values, edges_df['source'].values])
data = np.concatenate([edges_df['weight'].values, edges_df['weight'].values])
W_graph = sp.coo_matrix((data, (row, col)), shape=(R, R)).tocsr()
if verbose:
    print(f"  -> Matrice d'adjacence construite (W_graph). Densité : {W_graph.nnz / (R*R):.4%}")

v_W, val_W = solve_signed_spectral(W_graph, gpu=True)
pred_baseline = np.sign(v_W)
acc_baseline = np.mean(pred_baseline == true_spin_vec)
acc_baseline = max(acc_baseline, 1.0 - acc_baseline)
t_baseline = time.time() - t0
print(f"  -> Baseline Accuracy (vs true read spins): {acc_baseline:.4%} (Temps: {t_baseline:.2f}s)")

# 2. Glauber MCMC
print("⏳ Préparation de la dynamique de Glauber MCMC...")
t0 = time.time()
cross_offsets, cross_left, cross_right, cross_weight, \
incident_offsets, incident_left, incident_right, incident_weight = build_structures_fast(R, edges_df)

print(f"⏳ Génération des paires {k_hop}-hop...")
pairs_left, pairs_right = build_pairs_sparse(R, edges_df, k_hop)
P = len(pairs_left)
print(f"  -> Nombre de paires {k_hop}-hop : {P}")

# Paramètres MCMC
steps = mcmc_steps
# beta est défini dans le formulaire de configuration globale

tree = np.zeros(R, dtype=np.int32)

print(f"⏳ Lancement de la boucle MCMC ({steps} étapes)...")
t_mcmc_start = time.time()
correlations = mcmc_loop(steps, R, beta, tree,
                         cross_offsets, cross_left, cross_right, cross_weight,
                         incident_offsets, incident_left, incident_right, incident_weight,
                         pairs_left, pairs_right, verbose=verbose)
t_mcmc_duration = time.time() - t_mcmc_start
print(f"  -> Boucle MCMC terminée en {t_mcmc_duration:.2f}s.")

# 3. Spectral sur le Laplacien signé de la matrice de corrélation
print("⏳ Résolution spectrale sur la matrice de corrélation...")
row_C = np.concatenate([pairs_left, pairs_right])
col_C = np.concatenate([pairs_right, pairs_left])
data_C = np.concatenate([correlations, correlations])
W_C = sp.coo_matrix((data_C, (row_C, col_C)), shape=(R, R)).tocsr()
if verbose:
    print(f"  -> Matrice de corrélation construite (W_C). Éléments non nuls : {W_C.nnz}")

v_C, val_C = solve_signed_spectral(W_C, gpu=True)
pred_mcmc = np.sign(v_C)
acc_mcmc = np.mean(pred_mcmc == true_spin_vec)
acc_mcmc = max(acc_mcmc, 1.0 - acc_mcmc)
t_mcmc_total = time.time() - t0
print(f"  -> MCMC Accuracy (vs true read spins): {acc_mcmc:.4%} (Temps global: {t_mcmc_total:.2f}s)")


In [ ]:
# @title Reconstruction du Phasing des Variants et Export au format VCF
def write_phased_vcf(input_vcf_path, output_vcf_path, active_reads, read_id_to_idx, pred_spins, chromosome="chr22"):
    with pysam.VariantFile(input_vcf_path) as in_vcf:
        header = in_vcf.header
        with pysam.VariantFile(output_vcf_path, "w", header=header) as out_vcf:
            votes_by_pos = defaultdict(list)
            for rid, profile in active_reads.items():
                u = read_id_to_idx[rid]
                for pos, (allele_val, _) in profile.items():
                    votes_by_pos[pos].append((u, allele_val))
                    
            phased_count = 0
            for record in in_vcf:
                if record.chrom != chromosome:
                    continue
                pos = record.pos - 1
                if pos in votes_by_pos:
                    votes = votes_by_pos[pos]
                    vote_sum = 0.0
                    for u, allele_val in votes:
                        s_i = 1 - 2 * allele_val
                        vote_sum += s_i * pred_spins[u]
                    
                    sample_name = list(record.samples.keys())[0]
                    if vote_sum >= 0:
                        record.samples[sample_name]['GT'] = (0, 1)
                    else:
                        record.samples[sample_name]['GT'] = (1, 0)
                    record.samples[sample_name].phased = True
                    if 'PS' in in_vcf.header.formats:
                        ps_type = in_vcf.header.formats['PS'].type
                        record.samples[sample_name]['PS'] = '1' if ps_type == 'String' else 1
                    phased_count += 1
                else:
                    sample_name = list(record.samples.keys())[0]
                    record.samples[sample_name].phased = False
                    if 'PS' in in_vcf.header.formats:
                        record.samples[sample_name]['PS'] = None
                    
                out_vcf.write(record)
            return phased_count

# Réalignement des spins de prédiction par rapport à true_spin_vec pour l'écriture
best_baseline_spins = pred_baseline if np.mean(pred_baseline == true_spin_vec) >= 0.5 else -pred_baseline
best_mcmc_spins = pred_mcmc if np.mean(pred_mcmc == true_spin_vec) >= 0.5 else -pred_mcmc

print("⏳ Exportation des fichiers VCF phasés pour évaluation...")
c_base = write_phased_vcf("HG001_unphased.vcf", "phased_baseline.vcf", active_reads, read_id_to_idx, best_baseline_spins, chromosome)
c_mcmc = write_phased_vcf("HG001_unphased.vcf", "phased_mcmc.vcf", active_reads, read_id_to_idx, best_mcmc_spins, chromosome)

print(f"  -> Baseline VCF écrit (phased {c_base} variants).")
print(f"  -> MCMC VCF écrit (phased {c_mcmc} variants).")

In [ ]:
# @title Exécution de WhatsHap (SOTA) et Évaluation Comparative
with pysam.VariantFile("HG001_unphased.vcf") as vcf:
    sample_name = list(vcf.header.samples)[0]

print(f"⏳ Échantillon détecté dans le VCF : {sample_name}")
print("⏳ Exécution de WhatsHap pour le phasing de référence (uniquement sur chr22)...")
# WhatsHap va générer son propre phasing en utilisant le BAM et le VCF
!whatshap phase --chromosome {chromosome} --no-reference -o phased_whatshap.vcf HG001_unphased.vcf NA12878_chr22_alignment.bam

print("\n📊 Évaluation comparative par rapport au VCF de vérité terrain GIAB (whatshap compare)...")
print("1. Comparaison Baseline:")
!whatshap compare --sample {sample_name} HG001_phased_chr22.vcf phased_baseline.vcf > compare_baseline.txt
!cat compare_baseline.txt | grep -E "(switch errors|hamming distance|block n50)" || cat compare_baseline.txt | head -n 30

print("\n2. Comparaison MCMC:")
!whatshap compare --sample {sample_name} HG001_phased_chr22.vcf phased_mcmc.vcf > compare_mcmc.txt
!cat compare_mcmc.txt | grep -E "(switch errors|hamming distance|block n50)" || cat compare_mcmc.txt | head -n 30

print("\n3. Comparaison WhatsHap (État de l'art):")
!whatshap compare --sample {sample_name} HG001_phased_chr22.vcf phased_whatshap.vcf > compare_whatshap.txt
!cat compare_whatshap.txt | grep -E "(switch errors|hamming distance|block n50)" || cat compare_whatshap.txt | head -n 30

In [ ]:
# @title Extraction et Affichage des Métriques Finales
def parse_whatshap_compare(filepath):
    if not os.path.exists(filepath):
        print(f"⚠️ Warning: File {filepath} not found. Running compare might have failed!")
        return {"switch_errors": -1, "hamming_distance": -1, "block_n50": -1, "phased_variants": -1}
    metrics = {"switch_errors": 0, "hamming_distance": 0, "block_n50": 0, "phased_variants": 0}
    with open(filepath, "r") as f:
        for line in f:
            line_lower = line.lower()
            if "switch errors" in line_lower:
                parts = line.split()
                for p in parts:
                    if p.isdigit():
                        metrics["switch_errors"] = int(p)
            elif "hamming distance" in line_lower:
                parts = line.split()
                for p in parts:
                    if p.isdigit():
                        metrics["hamming_distance"] = int(p)
            elif "block n50" in line_lower:
                parts = line.split()
                for p in parts:
                    if p.replace(",", "").isdigit():
                        metrics["block_n50"] = int(p.replace(",", ""))
            elif "phased variants" in line_lower:
                parts = line.split()
                for p in parts:
                    if p.isdigit():
                        metrics["phased_variants"] = int(p)
    return metrics

print("⏳ Lecture et synthèse des résultats...")
m_baseline = parse_whatshap_compare("compare_baseline.txt")
m_mcmc = parse_whatshap_compare("compare_mcmc.txt")
m_whatshap = parse_whatshap_compare("compare_whatshap.txt")

df_summary = pd.DataFrame([
    {"Méthode": "Baseline Spectral (original)", **m_baseline},
    {"Méthode": "MCMC k-hop Spectral", **m_mcmc},
    {"Méthode": "WhatsHap (SOTA)", **m_whatshap}
])

print("\n" + "=" * 60)
print("TABLEAU COMPARATIF DES RÉSULTATS SUR NA12878 CHR22")
print("=" * 60)
print(df_summary.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

methods = ["Baseline", "MCMC", "WhatsHap"]
switches = [m_baseline["switch_errors"], m_mcmc["switch_errors"], m_whatshap["switch_errors"]]
n50s = [m_baseline["block_n50"], m_mcmc["block_n50"], m_whatshap["block_n50"]]

ax1.bar(methods, switches, color=['#FFA07A', '#4682B4', '#8FBC8F'], width=0.4)
ax1.set_ylabel("Nombre de Switch Errors")
ax1.set_title("Nombre d'erreurs de commutation (plus bas = meilleur)")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.bar(methods, n50s, color=['#FFA07A', '#4682B4', '#8FBC8F'], width=0.4)
ax2.set_ylabel("Block N50 (pb)")
ax2.set_title("Contiguïté du bloc de phase N50 (plus haut = meilleur)")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()